In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.stats import ttest_ind, chi2_contingency, pearsonr

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
fmt_k = mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}k ₹' if x >= 1000 else f'{int(x)} ₹')


In [8]:
df = pd.read_csv("../DATA/Clean_Dataset.csv")

df.columns = df.columns.str.strip().str.lower()

## Formulation et test d’hypothèses


### Sur la durée du vol et le  prix

#### Hypothèses 

| | Formulation |
|---|---|
| **H₀** | Il n'existe **aucune** corrélation linéaire entre la durée et le prix → ρ = 0 |
| **H₁** | Il existe une corrélation linéaire significative entre la durée et le prix → ρ ≠ 0 |

- Test **bilatéral** (on ne présuppose pas le signe de la corrélation)
- Seuil de signification : **α = 0.05**

In [9]:
from scipy.stats import pearsonr

r, p = pearsonr(df['duration'], df['price'])

alpha = 0.05

print(f'r de Pearson : {r:.4f}')
print(f'p-value      : {p:.2e}')
print(f'r²           : {r**2:.4f}  →  la durée explique {r**2*100:.1f}% de la variance du prix')
print()
if p < alpha:
    print(f'Décision (α={alpha}) : On REJETTE H₀ — corrélation statistiquement significative.')
else:
    print(f'Décision (α={alpha}) : On NE rejette PAS H₀ — pas de corrélation détectée.')

r de Pearson : 0.2042
p-value      : 0.00e+00
r²           : 0.0417  →  la durée explique 4.2% de la variance du prix

Décision (α=0.05) : On REJETTE H₀ — corrélation statistiquement significative.


### Interprétation du test

**r ≈ +0.20** : corrélation positive **faible**. La direction est claire (plus le vol est long, plus le billet tend à coûter cher), mais l'intensité est modeste.

**p-value ≪ 0.05** : avec 300 000 observations, le test a une puissance statistique très élevée — même une corrélation très faible devient significative. Le rejet de H₀ signifie que la corrélation n'est **pas due au hasard**, pas qu'elle est forte.

**r² ≈ 4%** : la durée à elle seule n'explique que 4% de la variabilité du prix. Les 96% restants sont attribuables à d'autres facteurs (`class`, `airline`, `days_left`, etc.).

> **Nuance importante :** Significativité statistique ≠ importance pratique. Avec n=300 000, un r de 0.05 serait déjà significatif. Ici r=0.20 est significatif ET modérément utile pour la prédiction, mais insuffisant seul.

## Sur les jours avants vole ou le billet a etait acheter

In [10]:
group_early = df[df["days_left"] > 10]["price"]
group_late = df[df["days_left"]<=10 ]["price"]
p_value = ttest_ind(group_early, group_late)
print("P-value:", p_value)

P-value: TtestResult(statistic=np.float64(-55.90225436682767), pvalue=np.float64(0.0), df=np.float64(300151.0))


### Interprétation du test t de Student
#### Le test t de Student a été appliqué afin de comparer les prix des billets d’avion entre deux groupes :

##### Group_early : billets achetés plus de 10 jours avant le départ
##### Group_late : billets achetés 10 jours ou moins avant le départ

##### Le résultat obtenu est :

##### Statistique t = -55.90
##### p-value = 0.0

##### Analyse des résultats

##### La p-value = 0.0 est largement inférieure au seuil de signification (α = 0.05).

####  Cela signifie que :

#### - On rejette l’hypothèse nulle (H₀)
#### -Il existe une différence statistiquement significative entre les deux groupes

#### La statistique t est négative (-55.90), ce qui indique que :

#### Le prix moyen des billets achetés tôt (group_early) est plus faible que celui des billets achetés tard (group_late)

---
### Test-Anova sur la facteur de airline avec le prix

### 1 - Seul les voles économie, puisqu'on sait deja que le business est plus chers.

## Hypothèses

**H₀ (Hypothèse nulle) :**
Les moyennes des prix sont égales pour toutes les compagnies aériennes..

**H₁ (Hypothèse alternative) :**
Au moins une compagnie aérienne présente une moyenne de prix significativement différente des autres.

## Conditions d'application
- Les observations sont **indépendantes** entre les groupes
- La variable dépendante (prix) suit approximativement une **loi normale** au sein de chaque groupe

### Seuil de signification
$$\alpha = 0.05$$

## Règle de décision
- Si $p\text{-value} < \alpha$ → **Rejet de H₀** : la compagnie aérienne influence significativement le prix
- Si $p\text{-value} \geq \alpha$ → **Non-rejet de H₀** : pas de différence significative entre les compagnies

In [7]:
economy_df = df[df['class'] == 'Economy']

groups = []

for airline, group in economy_df.groupby('airline'):
    groups.append(group['price'].values)

f_stat, p_value = stats.f_oneway(*groups,equal_var=False)

print("F-statistic: ",f_stat)
print("P-value: ",p_value)

F-statistic:  5713.58602785617
P-value:  0.0


### Avec une F-statistique de 5713.59 et une p-value ≈ 0.0, on rejette l'hypothèse nulle H₀.


### Sur le nombre Des arret

In [26]:
from scipy import stats

zero = df[df['stops'] == 'zero']['price']
one = df[df['stops'] == 'one']['price']
two_or_more = df[df['stops'] == 'two_or_more']['price']

# Test ANOVA
f_stat, p_value = stats.f_oneway(zero, one, two_or_more)

print("F-statistic:", f_stat)
print("p-value:", p_value)

alpha = 0.05
if p_value < alpha:
    print("Le nombre de stops influence significativement le prix.")
else:
    print("Pas d'effet significatif du nombre de stops.")

F-statistic: 6477.130362486214
p-value: 0.0
Le nombre de stops influence significativement le prix.
